# Mask2Former — fine-tune on DIVA-HisDB (CS18 / CS863 / CB55)



Trains the official `facebookresearch/Mask2Former` on any of the three DIVA-HisDB subsets, with either an R50 or a DINOv2 backbone. Assumes the prepared `Mask2Former/` folder and the `DIVA-HisDB/` data folder are already on Drive.



**Runtime:** GPU (T4 / L4 / A100). Pick via `Runtime → Change runtime type`.


## 1  Pick dataset + backbone


In [ ]:
DATASET   = 'cs863'    # 'cs18' | 'cs863' | 'cb55'

BACKBONE  = 'R50'     # 'R50'  | 'dinov2'



M2F_DRIVE    = '/content/drive/MyDrive/fsl-tl-manuscripts/50_modelling/Mask2Former'

DIVA_DRIVE   = '/content/drive/MyDrive/fsl-tl-manuscripts/00_data/DIVA-HisDB'

OUTPUT_DRIVE = f'/content/drive/MyDrive/fsl-tl-manuscripts/80_models/mask2former_{DATASET}_{BACKBONE}'



CONFIG_REL   = f'configs/{DATASET}/maskformer2_{BACKBONE}_{DATASET}.yaml'

OUTPUT_LOCAL = f'/content/output/{DATASET}_{BACKBONE}'


## 2  Mount Drive + mirror to local SSD

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil

from pathlib import Path



M2F_LOCAL  = Path('/content/Mask2Former')

DIVA_LOCAL = Path('/content/DIVA-HisDB')



if not M2F_LOCAL.exists():

    print('copying Mask2Former → /content …')

    shutil.copytree(M2F_DRIVE, M2F_LOCAL)

if not DIVA_LOCAL.exists():

    print('copying DIVA-HisDB → /content …')

    shutil.copytree(DIVA_DRIVE, DIVA_LOCAL)

print('ready:', M2F_LOCAL, DIVA_LOCAL)

assert (M2F_LOCAL / CONFIG_REL).exists(), f'missing config: {CONFIG_REL}'


## 3  Install deps + recompile MSDeformAttn for Colab's GPU

In [ ]:
!pip install -q cython scipy shapely timm h5py submitit scikit-image opencv-python

!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


In [ ]:
import os, subprocess, sys, shutil
ops_dir = M2F_LOCAL / 'mask2former/modeling/pixel_decoder/ops'
shutil.rmtree(ops_dir / 'build', ignore_errors=True)
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.0;7.5;8.0;8.6;8.9'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '.'],
    cwd=ops_dir, check=True,
)

In [ ]:
import torch, MultiScaleDeformableAttention, detectron2  # noqa: F401

print('torch       :', torch.__version__, '| cuda', torch.version.cuda)

print('detectron2  :', detectron2.__version__)

print('GPU         :', torch.cuda.get_device_name(0))

print('MSDeformAttn: OK')


## 4  Train



`DIVA_ROOT` env var tells `train_diva.py` where to find the data; `OUTPUT_DIR` overrides the config's default checkpoint path.


In [ ]:
%load_ext tensorboard

%tensorboard --logdir {OUTPUT_LOCAL}


In [ ]:
import os

os.environ['DIVA_ROOT'] = str(DIVA_LOCAL)

os.chdir(M2F_LOCAL)

!python train_diva.py \

    --config-file {CONFIG_REL} \

    --num-gpus 1 \

    OUTPUT_DIR {OUTPUT_LOCAL}


## 5  Sync trained model back to Drive


In [ ]:
import shutil

from pathlib import Path

out_drive = Path(OUTPUT_DRIVE)

out_drive.parent.mkdir(parents=True, exist_ok=True)

if out_drive.exists():

    shutil.rmtree(out_drive)

shutil.copytree(OUTPUT_LOCAL, out_drive)

print('synced →', out_drive)


## 6  Inference on a random test page


In [ ]:
import json, random, sys, cv2

from pathlib import Path

import matplotlib.pyplot as plt



sys.path.insert(0, str(M2F_LOCAL))

from detectron2.config import get_cfg

from detectron2.engine import DefaultPredictor

from detectron2.data import MetadataCatalog

from detectron2.projects.deeplab import add_deeplab_config

from detectron2.utils.visualizer import Visualizer, ColorMode

from mask2former import add_maskformer2_config

from mask2former.modeling.backbone import add_dinov2_backbone_config

from train_diva import register_diva, DIVA_SUBSETS



SCORE_THRESH = 0.5

register_diva()

metadata = MetadataCatalog.get(f'{DATASET}_test')



cfg = get_cfg()

add_deeplab_config(cfg); add_maskformer2_config(cfg); add_dinov2_backbone_config(cfg)

cfg.merge_from_file(str(M2F_LOCAL / CONFIG_REL))

cfg.MODEL.WEIGHTS = f'{OUTPUT_LOCAL}/model_final.pth'

cfg.MODEL.MASK_FORMER.TEST.OBJECT_MASK_THRESHOLD = SCORE_THRESH

cfg.freeze()

predictor = DefaultPredictor(cfg)



_, img_subdir = DIVA_SUBSETS[DATASET]

img_dir   = DIVA_LOCAL / img_subdir / 'public-test'

img_path  = random.choice(sorted(img_dir.glob('*.jpg')))

img       = cv2.imread(str(img_path))



inst = predictor(img)['instances'].to('cpu')

inst = inst[inst.scores >= SCORE_THRESH]

print(f'{img_path.name}: {len(inst)} text-lines (scores '

      f'min={float(inst.scores.min()):.3f} max={float(inst.scores.max()):.3f})')



viz = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.5,

                 instance_mode=ColorMode.IMAGE)

result = viz.draw_instance_predictions(inst).get_image()



out_local = Path(OUTPUT_LOCAL) / f'pred_{img_path.stem}.jpg'

cv2.imwrite(str(out_local), result[:, :, ::-1])

out_drive_img = Path(OUTPUT_DRIVE) / out_local.name

out_drive_img.parent.mkdir(parents=True, exist_ok=True)

cv2.imwrite(str(out_drive_img), result[:, :, ::-1])

print(f'saved → {out_local}\nsaved → {out_drive_img}')



plt.figure(figsize=(10, 14))

plt.imshow(result); plt.axis('off'); plt.show()
